# P4 — Mood Music Recommendation Integration
**Team G16 | Seungjae Chung**

P2 MERT-based valence/arousal predictions and P3 frozen CLAP audio–text scores are joined by `song_id`. Ranking weights are selected using development data only; the held-out test set is not used for tuning or reporting results.

## Data and configuration

Inputs: `p2_va_for_p4.csv`, `p3_clap_for_p4.csv`, `p3_clap_config.json`, and `deam_combined_500.csv` (reference valence/arousal labels for development-only evaluation). Input files are resolved from the working directory or Colab upload interface.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

FILES = ['p2_va_for_p4.csv', 'p3_clap_for_p4.csv', 'p3_clap_config.json', 'deam_combined_500.csv']
missing = [name for name in FILES if not Path(name).is_file()]
if missing:
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        pass
missing = [name for name in FILES if not Path(name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing input files: {missing}')

p2 = pd.read_csv(FILES[0]); p3 = pd.read_csv(FILES[1]); truth = pd.read_csv(FILES[3])
with open(FILES[2], encoding='utf-8') as f:
    config = json.load(f)
MOODS = config['moods']; TOP_K = 10
CLAP_COLS = [f'clap_{m}' for m in MOODS]
assert all(col in p2 for col in ['song_id','split','valence','arousal'])
assert all(col in p3 for col in ['song_id','split',*CLAP_COLS])
assert p2.song_id.is_unique and p3.song_id.is_unique
assert set(p2.song_id) == set(p3.song_id)
assert p2.set_index('song_id')['split'].sort_index().equals(p3.set_index('song_id')['split'].sort_index())
songs = p2.merge(p3.drop(columns='split'), on='song_id', validate='one_to_one')
assert songs[CLAP_COLS + ['valence','arousal']].apply(pd.to_numeric, errors='coerce').notna().all().all()
assert np.isfinite(songs[CLAP_COLS + ['valence','arousal']].to_numpy(dtype=float)).all()
assert songs.split.value_counts().to_dict() == {'train':350,'dev':75,'test':75}
print('P2/P3 aligned:',len(songs),'songs; splits:',songs.split.value_counts().to_dict())

Saving p3_clap_config.json to p3_clap_config.json
Saving complementarity_with_p2_dev.csv to complementarity_with_p2_dev.csv
Saving p3_clap_for_p4.csv to p3_clap_for_p4.csv
Saving p2_va_for_p4.csv to p2_va_for_p4.csv
Saving deam_combined_500.csv to deam_combined_500.csv
P2/P3 aligned: 500 songs; splits: {'train': 350, 'test': 75, 'dev': 75}


## Ranking

Emotion similarity is the normalized Euclidean proximity to the target valence/arousal point (1–9). CLAP similarity is min–max normalized independently for each mood over the candidate split. The final score is a weighted combination. Ties are broken by song ID.

In [2]:
def minmax(x):
    x = np.asarray(x, dtype=float)
    return np.full(len(x), 0.5) if np.isclose(x.max(),x.min()) else (x-x.min())/(x.max()-x.min())

def emotion_similarity(df, mood, valence_col='valence', arousal_col='arousal'):
    target = MOODS[mood]
    v = (df[valence_col].to_numpy(dtype=float)-target['valence'])/8
    a = (df[arousal_col].to_numpy(dtype=float)-target['arousal'])/8
    return 1 - np.sqrt(v*v+a*a)/np.sqrt(2)

def recommend(df, mood, weight=0.5, split='dev', k=TOP_K):
    if mood not in MOODS: raise ValueError(f'Unknown mood: {mood}')
    if not 0 <= weight <= 1: raise ValueError('weight must be between 0 and 1')
    if not isinstance(k,int) or k < 1: raise ValueError('k must be a positive integer')
    if split not in ('train','dev','test'): raise ValueError('Invalid split')
    candidates = df.loc[df.split == split].copy()
    if len(candidates) < k: raise ValueError('Not enough candidates')
    candidates['emotion_similarity'] = emotion_similarity(candidates,mood)
    candidates['clap_similarity'] = minmax(candidates[f'clap_{mood}'])
    candidates['combined_score'] = weight*candidates.emotion_similarity + (1-weight)*candidates.clap_similarity
    candidates['mood'] = mood
    return candidates.sort_values(['combined_score','song_id'],ascending=[False,True]).head(k).reset_index(drop=True)[['song_id','mood','valence','arousal',f'clap_{mood}','emotion_similarity','clap_similarity','combined_score']]


## Development-only weight selection

Select a single global weight by maximizing mean Spearman rank correlation between the combined recommendation score and the *true* valence/arousal proximity on the development split. This metric uses DEAM emotion labels, not activity-specific ground truth. Ties favor the larger emotion weight. The test split is not accessed for tuning.

In [3]:
labels = truth[['song_id','valence_mean','arousal_mean']].rename(columns={'valence_mean':'true_valence','arousal_mean':'true_arousal'})
assert labels.song_id.is_unique
dev = songs[songs.split=='dev'].merge(labels,on='song_id',validate='one_to_one')
assert len(dev)==75 and dev[['true_valence','true_arousal']].notna().all().all()
weights = np.round(np.linspace(0,1,21),2)
rows=[]
for w in weights:
    correlations=[]
    for mood in MOODS:
        predicted = w*emotion_similarity(dev,mood) + (1-w)*minmax(dev[f'clap_{mood}'])
        actual = emotion_similarity(dev,mood,'true_valence','true_arousal')
        correlations.append(pd.Series(predicted).corr(pd.Series(actual),method='spearman'))
    rows.append({'emotion_weight':w,'mean_dev_spearman':float(np.mean(correlations)),**{f'rho_{m}':v for m,v in zip(MOODS,correlations)}})
weight_results=pd.DataFrame(rows)
best=weight_results.sort_values(['mean_dev_spearman','emotion_weight'],ascending=[False,False]).iloc[0]
BEST_WEIGHT=float(best.emotion_weight)
print('Selected development-only emotion weight:',BEST_WEIGHT)
display(weight_results.sort_values('mean_dev_spearman',ascending=False).head(10))

Selected development-only emotion weight: 0.9


,emotion_weight,mean_dev_spearman,rho_Relax,rho_Study,rho_Workout,rho_Happy,rho_Sad
18,0.90,0.675365,0.397499,0.493328,0.818546,0.840660,0.826789
19,0.95,0.672082,0.394355,0.504965,0.818532,0.821955,0.820601
17,0.85,0.669174,0.392278,0.468804,0.816043,0.847133,0.821611
20,1.00,0.660013,0.377612,0.500185,0.807778,0.802851,0.811639
16,0.80,0.655003,0.369987,0.437779,0.802302,0.854117,0.810828
15,0.75,0.640920,0.359091,0.406171,0.791932,0.844714,0.802691
14,0.70,0.623141,0.346587,0.369029,0.780068,0.830902,0.789121
13,0.65,0.605798,0.331750,0.343779,0.764534,0.820319,0.768608
12,0.60,0.588663,0.318093,0.321745,0.744136,0.809607,0.749732
11,0.55,0.572975,0.312716,0.298970,0.724448,0.798014,0.730727


## Interactive recommendation interface

The dropdown displays the top ten recommendations from the development partition using the selected global weight.

In [4]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    mood_dropdown=widgets.Dropdown(options=list(MOODS),description='Mood:')
    out=widgets.Output()
    def update(change=None):
        with out:
            clear_output(wait=True)
            display(recommend(songs,mood_dropdown.value,weight=BEST_WEIGHT))
    mood_dropdown.observe(update,names='value')
    display(mood_dropdown,out)
    update()
except ImportError:
    display(recommend(songs,'Relax',weight=BEST_WEIGHT))

Dropdown(description='Mood:', options=('Relax', 'Study', 'Workout', 'Happy', 'Sad'), value='Relax')

Output()

## Automated checks and development recommendations

In [5]:
for mood in MOODS:
    ranked=recommend(songs,mood,weight=BEST_WEIGHT)
    assert len(ranked)==TOP_K
    assert ranked.song_id.is_unique
    assert ranked.combined_score.is_monotonic_decreasing
    assert set(ranked.song_id).issubset(set(dev.song_id))
for invalid in ['Invalid','']:
    try: recommend(songs,invalid)
    except ValueError: pass
    else: raise AssertionError('Invalid mood accepted')
try: recommend(songs,'Relax',weight=1.1)
except ValueError: pass
else: raise AssertionError('Invalid weight accepted')
results=pd.concat([recommend(songs,mood,weight=BEST_WEIGHT) for mood in MOODS],ignore_index=True)
results.to_csv('P4_P2_P3_DEV_top10.csv',index=False)
weight_results.to_csv('P4_dev_weight_comparison.csv',index=False)
print('Tests passed; exported',len(results),'development recommendations across',len(MOODS),'moods.')
display(results.groupby('mood').size().rename('top_k'))

Tests passed; exported 50 development recommendations across 5 moods.


,top_k
mood,
Happy,10
Relax,10
Sad,10
Study,10
Workout,10


## Evaluation scope

P2 and P3 are real model outputs. The global weight is tuned on development labels only; the test split remains reserved for final evaluation. DEAM provides valence/arousal annotations, not direct activity labels. Development metrics used for model selection are not unbiased estimates of final performance.